# Revenue to City-Mapping Mismatch Report

Validates that all airport codes (FromAirport, ToAirport) have city mappings via displayCode.

In [9]:
import pandas as pd
import numpy as np

# Load datasets
mapping_df = pd.read_excel('gs://agntworks-data-dev/wheelsup/raw/Mappings/ICAO to City Mapping.xlsx')
revenue_df = pd.read_parquet('gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet')

# Define Wheels Up operators
WU_OPERATORS = [
    'Wheels Up Private Jets', 'Wheels Up LLC',
    'Mountain Aviation', 'Alante Air Charter'
]

print("✓ Data loaded successfully!")
print(f"Revenue records: {len(revenue_df)}")
print(f"Mapping records: {len(mapping_df)}")

✓ Data loaded successfully!
Revenue records: 2140706
Mapping records: 30669


In [10]:
# Extract mapping data
mapped_codes = set(mapping_df['displayCode'].dropna().unique())
mapped_cities = mapping_df.groupby('city')['displayCode'].apply(lambda x: sorted(x.dropna().unique())).to_dict()

print(f"=== MAPPING DATA SUMMARY ===")
print(f"Total unique displayCodes: {len(mapped_codes)}")
print(f"Total unique cities: {len(mapped_cities)}")

=== MAPPING DATA SUMMARY ===
Total unique displayCodes: 27115
Total unique cities: 13277


In [11]:
# Pre-compute airport counts for speed
from_counts_all = revenue_df['FromAirport'].value_counts()
to_counts_all = revenue_df['ToAirport'].value_counts()

wu_revenue_df = revenue_df[revenue_df['Operator'].isin(WU_OPERATORS)]
from_counts_wu = wu_revenue_df['FromAirport'].value_counts()
to_counts_wu = wu_revenue_df['ToAirport'].value_counts()

print("✓ Pre-computed airport counts")

✓ Pre-computed airport counts


In [12]:
# === SECTION 1: ALL ENTITIES ===
print(f"\n" + "="*70)
print("1. ALL ENTITIES (Filtered 2024-2025)")
print("="*70)

all_from_airports = set(from_counts_all.index)
all_to_airports = set(to_counts_all.index)
all_revenue_airports = all_from_airports | all_to_airports

all_missing = all_revenue_airports - mapped_codes

print(f"\nUnique Origin ICAOs: {len(all_revenue_airports)}")
print(f"✗ Missing from Mapping: {len(all_missing)}")


1. ALL ENTITIES (Filtered 2024-2025)

Unique Origin ICAOs: 2631
✗ Missing from Mapping: 621


In [13]:
# Details of missing mappings for ALL ENTITIES (vectorized)
if all_missing:
    print(f"\nMissing airports:")
    missing_details = []
    for airport in all_missing:
        from_count = from_counts_all.get(airport, 0)
        to_count = to_counts_all.get(airport, 0)
        total = from_count + to_count
        missing_details.append({
            'Airport': airport,
            'From Records': from_count,
            'To Records': to_count,
            'Total': total
        })

    missing_df = pd.DataFrame(missing_details)
    missing_df = missing_df.sort_values('Total', ascending=False)
    print(f"\n{missing_df.to_string(index=False)}")
    print(f"\nTotal records affected: {missing_df['Total'].sum()}")


Missing airports:

Airport  From Records  To Records  Total
   KT82          1667        1727   3394
   K5B2           746         767   1513
   K0A9           718         726   1444
   K27K           720         724   1444
   KF45           611         594   1205
   K29D           590         605   1195
   K1R8           557         533   1090
   K1A5           441         444    885
   KF82           456         421    877
   K5C1           421         427    848
   KX60           379         452    831
   K11R           387         397    784
   K8A0           351         318    669
   K46U           306         320    626
   KM19           262         291    553
   K4I3           264         272    536
   KO69           247         270    517
   K20V           253         256    509
   K38S           238         247    485
   K3T5           231         242    473
   KS25           227         238    465
   KS21           228         207    435
   KE38           217         201    

In [14]:
# === SECTION 2: TARGET WHEELS UP ENTITIES ===
print(f"\n" + "="*70)
print(f"2. TARGET WHEELS UP ENTITIES (4 Operators)")
print("="*70)

print(f"\nWheels Up operators:")
for op in WU_OPERATORS:
    count = len(wu_revenue_df[wu_revenue_df['Operator'] == op])
    print(f"  {op}: {count} records")

wu_from_airports = set(from_counts_wu.index)
wu_to_airports = set(to_counts_wu.index)
wu_revenue_airports = wu_from_airports | wu_to_airports

wu_missing = wu_revenue_airports - mapped_codes

print(f"\nUnique Origin ICAOs: {len(wu_revenue_airports)}")
print(f"✗ Missing from Mapping: {len(wu_missing)}")

if len(wu_revenue_airports) > 0:
    wu_coverage_pct = ((len(wu_revenue_airports) - len(wu_missing)) / len(wu_revenue_airports)) * 100
    print(f"Coverage: {wu_coverage_pct:.1f}%")


2. TARGET WHEELS UP ENTITIES (4 Operators)

Wheels Up operators:
  Wheels Up Private Jets: 47791 records
  Wheels Up LLC: 0 records
  Mountain Aviation: 19258 records
  Alante Air Charter: 4001 records

Unique Origin ICAOs: 1159
✗ Missing from Mapping: 50
Coverage: 95.7%


In [15]:
# Details of missing mappings for WHEELS UP ENTITIES (vectorized)
if wu_missing:
    print(f"\nMissing airports:")
    wu_missing_details = []
    for airport in wu_missing:
        from_count = from_counts_wu.get(airport, 0)
        to_count = to_counts_wu.get(airport, 0)
        total = from_count + to_count
        wu_missing_details.append({
            'Airport': airport,
            'From Records': from_count,
            'To Records': to_count,
            'Total': total
        })

    wu_missing_df = pd.DataFrame(wu_missing_details)
    wu_missing_df = wu_missing_df.sort_values('Total', ascending=False)
    print(f"\n{wu_missing_df.to_string(index=False)}")
    print(f"\nTotal records affected: {wu_missing_df['Total'].sum()}")


Missing airports:

Airport  From Records  To Records  Total
   KT82            80          84    164
   K5B2            41          39     80
   K0A9            20          26     46
   K1A5            10          14     24
   KE38            12          11     23
   K2I0             8          10     18
   KS21            10           8     18
   K4I3             9           9     18
   K2G4             7          10     17
   K11R             8           8     16
   KS24             9           7     16
   KF45             7           8     15
   K20V             8           6     14
   K38S             6           6     12
   KMO8             5           5     10
   K1H2             4           5      9
   K3T5             5           4      9
   K1B1             5           4      9
   K8A1             4           4      8
   K6A1             3           5      8
   K4R3             4           4      8
   KX60             3           3      6
   K27K             3           3    

In [16]:
# === SECTION 3: SUMMARY ===
print(f"\n" + "="*70)
print(f"SUMMARY")
print("="*70)

print(f"\nAll Entities:")
print(f"  Total airports used: {len(all_revenue_airports)}")
print(f"  Airports with mapping: {len(all_revenue_airports - all_missing)}")
print(f"  Airports missing mapping: {len(all_missing)}")

print(f"\nWheels Up Entities:")
print(f"  Total airports used: {len(wu_revenue_airports)}")
print(f"  Airports with mapping: {len(wu_revenue_airports - wu_missing)}")
print(f"  Airports missing mapping: {len(wu_missing)}")


SUMMARY

All Entities:
  Total airports used: 2631
  Airports with mapping: 2010
  Airports missing mapping: 621

Wheels Up Entities:
  Total airports used: 1159
  Airports with mapping: 1109
  Airports missing mapping: 50
